# 01 – Data exploration and CSP + LDA baseline

Euclidean baseline for 4-class motor imagery on BCI Competition IV 2a.

- Data pipeline and design choices: `src/riemannian_eeg/data.py` and `docs/design_decisions.md`.
- Why each trial is represented by a covariance (SPD) matrix: `docs/eeg_2_spd.md`.

In [ ]:
from riemannian_eeg.data import load_train_test

X_train, y_train, X_test, y_test = load_train_test(subject=1)
print(X_train.shape, X_test.shape)

## CSP + LDA baseline

CSP + LDA is the classic pipeline for motor imagery, so it's our reference point. It uses
the same information as the Riemannian methods (covariances between electrodes) but treats
them as if they lived in a flat, Euclidean space, so any accuracy gap with MDM, tangent
space or SPDNet comes from the geometry.

### Spatial filters as virtual electrodes

A spatial filter $w \in \mathbb{R}^{22}$ combines the 22 electrodes into one new signal, a
"virtual electrode", $s(t) = w^\top x(t)$. The key fact CSP builds on is that the power of any
virtual electrode can be computed directly from the trial's covariance matrix $C$:

$$
\operatorname{Var}(s) = w^\top C \, w.
$$

So $C$ describes the trial, and $w$ says how the electrodes are combined. The full derivation
(zero mean after band-pass filtering, trial covariance, the two-electrode example, and why $C$ is
SPD) is in `docs/eeg_2_spd.md`.

### The CSP objective

Applying the same filter $w$ to the trials of each class, $w^\top C_1 w$ is the power of the
virtual electrode for class 1 and $w^\top C_2 w$ its power for class 2, where $C_1$ and $C_2$ are
the average covariances of each class. CSP looks for the filter whose power is as large as
possible for class 1 and as small as possible for class 2, i.e. it maximizes their ratio:

$$
\max_w \; J(w) = \frac{w^\top C_1 w}{w^\top C_2 w}.
$$

**Only the direction of $w$ matters.** Rescaling $w$ by any $a \neq 0$ multiplies both powers by
$a^2$, which cancels out: $J(aw) = J(w)$. So we're free to pick the scale of $w$, and we choose
the one that makes the denominator equal to 1. The problem becomes a constrained one:

$$
\max_w \; w^\top C_1 w \quad \text{subject to} \quad w^\top C_2 w = 1.
$$

**Lagrange multipliers.** To maximize a function under a constraint, we build the Lagrangian,
which adds the constraint (set equal to zero) multiplied by a new variable $\lambda$:

$$
L(w, \lambda) = w^\top C_1 w - \lambda \left(w^\top C_2 w - 1\right).
$$

At the optimal point, the gradient of $L$ with respect to $w$ must be zero (otherwise we could
still move along the constraint and increase the power ratio). Since $C_1$ and $C_2$ are
symmetric, the derivative of a quadratic form is $\frac{\partial}{\partial w}(w^\top A w) = 2Aw$
(the matrix version of $\frac{d}{dx}(a x^2) = 2ax$), so:

$$
\frac{\partial L}{\partial w} = 2\,C_1 w - 2\lambda\, C_2 w = 0
\quad\Longrightarrow\quad
C_1 w = \lambda \, C_2 w.
$$

This is a **generalized eigenvalue problem**: the solutions $w$ are the generalized eigenvectors,
each with its eigenvalue $\lambda$.

**Meaning of $\lambda$.** Multiplying both sides by $w^\top$ on the left:

$$
w^\top C_1 w = \lambda \, w^\top C_2 w
\quad\Longrightarrow\quad
\lambda = \frac{w^\top C_1 w}{w^\top C_2 w} = J(w).
$$

So each eigenvalue is exactly the power ratio of its filter. Filters with the largest eigenvalues
have high power in class 1 and low in class 2, and those with the smallest eigenvalues the
opposite. We keep a few from each end and drop the middle ones, which don't discriminate.

**From each trial to a few features.** Solving the eigenvalue problem gives a few filters
$w_1, \dots, w_k$ (e.g. $k = 6$, taken from both ends of the spectrum). For each trial, we compute
the power of every virtual electrode with that trial's covariance $C$, and take its log:

$$
f_j = \log\!\left(w_j^\top C \, w_j\right), \quad j = 1, \dots, k.
$$

So a trial that was a $22 \times 1001$ matrix ends up summarized by just $k$ numbers, the features
that LDA receives. The log is there because powers are always positive and very skewed (most are
small, a few are huge); taking the log spreads them out and makes each class roughly Gaussian,
which is what LDA assumes.

**How many filters.** The number of filters $k$ is a key hyperparameter of CSP: too few can miss
discriminative directions, while too many add filters from the middle of the spectrum (power
ratio close to 1) that carry little class information and give LDA more parameters to estimate
from few trials. Common practice is to keep a few filters from each end of the spectrum, typically
three pairs ($k = 6$) in the two-class case. Rather than fixing $k$ by hand, we select it by cross-validation on the training session. In
One-vs-Rest each binary model is a standard two-class CSP, so we try $k \in \{4, 6\}$ filters per
model (two or three pairs). In multiclass CSP there are no "two ends" to take pairs from, since
components are ranked by how much class information they carry, so we try a wider range of total
filters, $k \in \{4, 6, 8, 10, 12\}$.

*Refs: Blankertz et al. (2008), IEEE Signal Process. Mag. 25(1); Yang, Chevallier, Wiart &
Bloch (2012), "Automatic selection of the number of spatial filters for motor-imagery BCI",
ESANN 2012.*

**Where CSP is Euclidean.** To solve the problem we need $C_1$ and $C_2$, the average covariance of
each class. CSP computes them as plain arithmetic means: add up the covariance matrices of all the
trials of a class and divide by their number, as if SPD matrices lived in a flat space. But
covariances measure power, which is always positive and is naturally compared through ratios
rather than differences. The arithmetic mean ignores this: it is pulled towards the trials with
the largest power, and the resulting average can look more spread out (larger determinant) than
the matrices it averages, the so-called swelling effect. The Riemannian methods instead average
along the curved SPD manifold (the Fréchet mean), which respects these ratios and avoids the
distortion. This averaging step is exactly what changes between CSP and the Riemannian methods,
which is why CSP is the right baseline to compare against.

### Physiological intuition

The motor cortex is organized by body part: the area controlling each part of the body lies at a
different position along it, and each hemisphere controls the opposite side of the body. When a
movement is imagined, the mu/beta power over the corresponding area drops (Event-Related
Desynchronization, ERD), and over other areas it can increase (Event-Related Synchronization, ERS).
Over the electrodes we have, this gives a characteristic pattern for each class:

| Imagined movement | Motor area involved | Where the power drops (ERD) |
|---|---|---|
| Left hand | Hand area, right hemisphere | Around C4 |
| Right hand | Hand area, left hemisphere | Around C3 |
| Feet | Foot area, on the midline between hemispheres | Around Cz, weaker because the area lies deep |
| Tongue | Face/tongue area, lateral and lower | Less focal; often ERS over the hand areas (C3, C4) |

So the hand classes are told apart by the left-right contrast between C3 and C4, while feet and
tongue are recognized by a central ERD (feet) or by the hand areas becoming *more* active
(ERS) instead of less. CSP isn't told any of this: it only receives the trials and their labels.
But since its filters look for directions where power differs between classes, they usually end
up weighting exactly these electrodes. Because it keeps filters from both ends of the eigenvalue
spectrum, it captures both power drops (ERD) and power increases (ERS).

*Refs: Ramoser, Müller-Gerking & Pfurtscheller (2000), IEEE Trans. Rehabil. Eng. 8(4);
Blankertz et al. (2008), IEEE Signal Process. Mag. 25(1);
Grosse-Wentrup & Buss (2008), "Multiclass common spatial patterns and information theoretic
feature extraction", IEEE Trans. Biomed. Eng. 55(8).*

In [ ]:
from mne.decoding import CSP

from riemannian_eeg.data import DATASET, PARADIGM

# CSP fitted on the training session of subject 1.
# n_components: number of spatial filters kept (virtual electrodes).
# log=True: each trial is summarized by the log-variance of its filtered signals.
# With 4 classes, MNE uses the multiclass version (joint diagonalization).
csp = CSP(n_components=8, log=True)
csp.fit(X_train, y_train)

# Spatial patterns on the scalp (channel positions come from the MNE epochs object)
epochs, _, _ = PARADIGM.get_data(dataset=DATASET, subjects=[1], return_epochs=True)
csp.plot_patterns(epochs.info, ch_type="eeg", units="Patterns (AU)", size=1.5)

### LDA

**LDA (Linear Discriminant Analysis)** is the classifier that receives the CSP features. Each
trial now is a small vector $f \in \mathbb{R}^k$ (its $k$ log-variances), and LDA has to decide
which of the 4 classes it belongs to.

**Model.** LDA assumes that the features of each class follow a Gaussian distribution, with its
own mean $\mu_k$ but the same covariance $\Sigma$ for all classes:

$$
f \mid \text{class } k \;\sim\; \mathcal{N}(\mu_k, \Sigma).
$$

Training just means estimating these from the training trials: $\mu_k$ is the average feature
vector of class $k$, and $\Sigma$ the covariance of the features around their class means.

**Decision rule.** For a new trial, LDA picks the most probable class given $f$ (Bayes' rule).
Taking logs of the Gaussian densities and dropping the terms that are equal for all classes,
the term $f^\top \Sigma^{-1} f$ cancels precisely because $\Sigma$ is shared, and what is left
is a score that is *linear* in $f$:

$$
\delta_k(f) = f^\top \Sigma^{-1} \mu_k - \tfrac{1}{2}\, \mu_k^\top \Sigma^{-1} \mu_k + \log \pi_k,
$$

where $\pi_k$ is the prior probability of class $k$ (equal here, since the classes are balanced).
The predicted class is the one with the highest score. Because each $\delta_k$ is linear, the
boundary between any two classes ($\delta_j = \delta_k$) is a hyperplane: LDA separates the
classes with straight cuts in feature space.

**Why it fits CSP.** The log makes the CSP features roughly Gaussian, which is LDA's main
assumption. LDA also has very few parameters (the class means and one shared covariance), so it
can be estimated reliably from the 288 training trials without overfitting, and it is the most
widely used classifier in motor imagery BCIs.

*Refs: Fisher (1936), "The use of multiple measurements in taxonomic problems", Annals of
Eugenics 7(2); Hastie, Tibshirani & Friedman (2009), The Elements of Statistical Learning,
Sec. 4.3; Lotte et al. (2007), "A review of classification algorithms for EEG-based
brain-computer interfaces", J. Neural Eng. 4(2).*

### From 2 to 4 classes

The CSP objective compares two classes ($C_1$ vs $C_2$), so with four classes it has to be
extended. We compare two standard options.

**One-vs-Rest (OvR).** We train 4 independent binary models, one per class. In the model for
class $k$, $C_1$ is the average covariance of class $k$ and $C_2$ the average covariance of the
other three classes pooled together. Each model learns its own CSP filters (the directions that
best separate *its* class from the rest) and its own LDA. To classify a new trial, the 4 models
give a score and we pick the class whose model is most confident.

**Multiclass CSP by joint diagonalization (MNE).** In the two-class case, the generalized
eigenvectors make $W^\top C_1 W$ and $W^\top C_2 W$ both diagonal: in the filtered space, each
virtual electrode has its own power, with no cross-terms between them. With four classes there is
in general no single $W$ that diagonalizes the four $C_k$ exactly, so MNE finds the $W$ that
makes them *as diagonal as possible* at the same time (approximate joint diagonalization). This
gives one shared set of filters for all classes, and MNE keeps the ones that carry the most
information about the class label (Grosse-Wentrup & Buss, 2008).

The trade-off: OvR gives each class filters tailored to it, but each model only sees a "one
class vs everything" contrast. Joint diagonalization uses a single set of filters fitted on all
four classes at once. We don't consider One-vs-One (one model per pair of classes), since it
would need 6 models, each trained on only half the data.

The variant is chosen by cross-validation on the training session only, so the test session
stays untouched until the final evaluation.